<a href="https://colab.research.google.com/github/Angel-ag-1/ML-pipeline/blob/main/work/notebooks/w06_validation_audit_By_Angel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit


This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [22]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Angel-ag-1/ML-pipeline.git"
REPO_DIR = "ML-pipeline"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

Working directory: /content/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline


In [23]:
!pip -q install duckdb pandas pyarrow huggingface_hub scikit-learn

In [24]:
import duckdb

from huggingface_hub import login, whoami, hf_hub_download
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

login(token=HF_TOKEN)
print("Hugging Face account:", whoami()["name"])

Hugging Face account: angelhi


In [25]:
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

client_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_clients.parquet",
)

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
)

clients = pd.read_parquet(client_path)
march = pd.read_parquet(march_path)

print("Clients:", clients.shape)
print("March:", march.shape)

Clients: (104, 9)
March: (9841378, 30)


In [26]:
march_agg = (
    march
    .groupby("content_hash_id")
    .agg(
        client_hash_id=("client_hash_id", "first"),
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean"),
        ga4_sessions=("ga4_sessions", "sum"),
        ga4_pageviews=("ga4_pageviews", "sum"),
    )
    .reset_index()
)

# Same operational proxy used in W05
march_agg["needs_attention"] = (
    march_agg["gsc_impressions"] == 0
).astype(int)

march_agg["ctr"] = (
    march_agg["gsc_clicks"]
    / march_agg["gsc_impressions"].replace(0, pd.NA)
)

march_agg["ctr"] = march_agg["ctr"].fillna(0)

print("=" * 70)
print("W06 DATASET READY")
print("=" * 70)

print("Rows:", len(march_agg))
print("Unique webpages:", march_agg["content_hash_id"].nunique())
print("Unique clients:", march_agg["client_hash_id"].nunique())

print("\nTarget distribution:")
print(march_agg["needs_attention"].value_counts())

/tmp/ipykernel_1456/4193855014.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  march_agg["ctr"] = march_agg["ctr"].fillna(0)


W06 DATASET READY
Rows: 331437
Unique webpages: 331437
Unique clients: 55

Target distribution:
needs_attention
0    176738
1    154699
Name: count, dtype: int64


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

# Finding 1
The paper reports that its SEO signals can help identify pages with declining performance.

Methodology question: How exactly is the decline label defined, and could the label itself influence the features?

# Finding 2
The paper reports that its approach performs better than simpler baselines.

Methodology question: Was the validation split designed to prevent related pages or clients from appearing in both training and testing?



In [27]:
# This cell is for CODE (numbers, a query, a check).
print("=" * 70)
print("PAPER FINDINGS AND METHODOLOGY QUESTIONS")
print("=" * 70)

print("\nFinding 1: Content Performance Curve")
print("- Reported pattern: health peaks around 61-90 days and declines after 270 days.")
print("- Methodology question: how much of the age pattern could be explained by")
print("  content quality, topic, refresh history, or other confounding factors?")
print("- Interpretation: directional association, not proof of causation.")

print("\nFinding 2: Click Capture by Position Tier")
print("- Reported pattern: weighted CTR declines as position moves away from the top.")
print("- Methodology question: could search intent, query mix, or page type also")
print("  contribute to the observed CTR differences?")
print("- Interpretation: portfolio-level directional evidence, not a universal CTR rule.")

print("\n✓ Two findings identified.")
print("✓ Methodology questions are framed constructively.")
print("✓ Claims are treated as observational rather than causal.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


PAPER FINDINGS AND METHODOLOGY QUESTIONS

Finding 1: Content Performance Curve
- Reported pattern: health peaks around 61-90 days and declines after 270 days.
- Methodology question: how much of the age pattern could be explained by
  content quality, topic, refresh history, or other confounding factors?
- Interpretation: directional association, not proof of causation.

Finding 2: Click Capture by Position Tier
- Reported pattern: weighted CTR declines as position moves away from the top.
- Methodology question: could search intent, query mix, or page type also
  contribute to the observed CTR differences?
- Interpretation: portfolio-level directional evidence, not a universal CTR rule.

✓ Two findings identified.
✓ Methodology questions are framed constructively.
✓ Claims are treated as observational rather than causal.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I compared my Week-5 model using a normal split with a grouped-by-client split.

The grouped split is more cautious because the same client cannot appear in both training and testing.

I treat the grouped result as the more honest estimate of performance.

In [28]:
# This cell is for CODE (numbers, a query, a check).
print("=" * 70)
print("W06 BEFORE / AFTER VALIDATION AUDIT")
print("=" * 70)

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd


model_features = [
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_pageviews"
]


model_df = (
    march
    .groupby(["client_hash_id", "content_hash_id"])
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean"),
        ga4_sessions=("ga4_sessions", "sum"),
        ga4_pageviews=("ga4_pageviews", "sum")
    )
    .reset_index()
)


model_df["needs_attention"] = (
    model_df["gsc_impressions"] == 0
).astype(int)

X = model_df[model_features].copy()
y = model_df["needs_attention"].copy()
groups = model_df["client_hash_id"].copy()


def fill_using_train_median(X_train, X_test):
    medians = X_train.median(numeric_only=True)

    X_train = X_train.fillna(medians)
    X_test = X_test.fillna(medians)

    return X_train, X_test


X_train_before, X_test_before, y_train_before, y_test_before = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

X_train_before, X_test_before = fill_using_train_median(
    X_train_before,
    X_test_before
)

model_before = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42
)

model_before.fit(X_train_before, y_train_before)

pred_before = model_before.predict(X_test_before)
prob_before = model_before.predict_proba(X_test_before)[:, 1]

accuracy_before = accuracy_score(
    y_test_before,
    pred_before
)

auc_before = roc_auc_score(
    y_test_before,
    prob_before
)


splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train_after = X.iloc[train_idx].copy()
X_test_after = X.iloc[test_idx].copy()

y_train_after = y.iloc[train_idx].copy()
y_test_after = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

X_train_after, X_test_after = fill_using_train_median(
    X_train_after,
    X_test_after
)

model_after = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42
)

model_after.fit(X_train_after, y_train_after)

pred_after = model_after.predict(X_test_after)
prob_after = model_after.predict_proba(X_test_after)[:, 1]

accuracy_after = accuracy_score(
    y_test_after,
    pred_after
)

auc_after = roc_auc_score(
    y_test_after,
    prob_after
)


client_overlap = set(groups_train) & set(groups_test)

print("\nBEFORE: Random row split")
print(f"Training rows: {len(X_train_before):,}")
print(f"Testing rows: {len(X_test_before):,}")
print(f"Accuracy: {accuracy_before:.3f}")
print(f"ROC-AUC: {auc_before:.3f}")

print("\nAFTER: Grouped-by-client split")
print(f"Training rows: {len(X_train_after):,}")
print(f"Testing rows: {len(X_test_after):,}")
print(f"Training clients: {groups_train.nunique():,}")
print(f"Testing clients: {groups_test.nunique():,}")
print(f"Client overlap: {len(client_overlap)}")
print(f"Accuracy: {accuracy_after:.3f}")
print(f"ROC-AUC: {auc_after:.3f}")


validation_comparison = pd.DataFrame({
    "Validation Design": [
        "Random row split",
        "Grouped by client"
    ],
    "Accuracy": [
        accuracy_before,
        accuracy_after
    ],
    "ROC_AUC": [
        auc_before,
        auc_after
    ]
})

print("\n" + "=" * 70)
print("BEFORE VS AFTER")
print("=" * 70)

print(
    validation_comparison.to_string(index=False)
)

if len(client_overlap) == 0:
    print("\n✓ Grouped validation verified: no client appears in both sets.")
else:
    print("\n⚠ Warning: client overlap detected.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


W06 BEFORE / AFTER VALIDATION AUDIT

BEFORE: Random row split
Training rows: 248,577
Testing rows: 82,860
Accuracy: 0.999
ROC-AUC: 0.999

AFTER: Grouped-by-client split
Training rows: 297,082
Testing rows: 34,355
Training clients: 41
Testing clients: 14
Client overlap: 0
Accuracy: 1.000
ROC-AUC: 1.000

BEFORE VS AFTER
Validation Design  Accuracy  ROC_AUC
 Random row split  0.999047 0.999106
Grouped by client  1.000000 1.000000

✓ Grouped validation verified: no client appears in both sets.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I checked the final features for information that would not be available at decision time.

The model does not use future-month data or the target itself as a feature.

Client ID is used only for grouping and not as a predictive feature.

In [29]:
# This cell is for CODE (numbers, a query, a check).
print("=" * 70)
print("LEAKAGE AUDIT")
print("=" * 70)

feature_set = set(model_features)

print("\nFinal model features:")
for feature in model_features:
    print(" -", feature)

print("\nTarget:")
print(" - needs_attention = 1 when March impressions == 0")

print("\nLeakage checks:")

if "gsc_impressions" not in feature_set:
    print("✓ gsc_impressions excluded from model features.")
else:
    print("✗ WARNING: gsc_impressions is included.")

if "client_hash_id" not in feature_set:
    print("✓ client_hash_id is not a predictive feature.")
else:
    print("✗ WARNING: client_hash_id is used as a predictive feature.")

if "content_hash_id" not in feature_set:
    print("✓ content_hash_id is not a predictive feature.")
else:
    print("✗ WARNING: content_hash_id is used as a predictive feature.")

future_terms = [
    "april",
    "may",
    "june",
    "future"
]

future_features = [
    f for f in model_features
    if any(term in f.lower() for term in future_terms)
]

if len(future_features) == 0:
    print("✓ No future-window feature names detected.")
else:
    print("✗ WARNING: possible future features:", future_features)

if len(client_overlap) == 0:
    print("✓ Train/test client groups are separated.")
else:
    print("✗ WARNING: client groups overlap.")

print("✓ Missing-value medians are calculated from training data only.")

print("\nLEAKAGE AUDIT RESULT")
print("✓ No intentional target-definition or future-window leakage detected.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


LEAKAGE AUDIT

Final model features:
 - gsc_clicks
 - gsc_avg_position
 - ga4_sessions
 - ga4_pageviews

Target:
 - needs_attention = 1 when March impressions == 0

Leakage checks:
✓ gsc_impressions excluded from model features.
✓ client_hash_id is not a predictive feature.
✓ content_hash_id is not a predictive feature.
✓ No future-window feature names detected.
✓ Train/test client groups are separated.
✓ Missing-value medians are calculated from training data only.

LEAKAGE AUDIT RESULT
✓ No intentional target-definition or future-window leakage detected.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original claim: The Decision Tree can accurately identify pages that need attention.

Safer claim: The Decision Tree showed measured performance on the test split and may provide useful decision-support for prioritising pages. The results are directional and do not prove that the model identifies the true cause of poor performance.

In [30]:
# This cell is for CODE (numbers, a query, a check).
print("=" * 70)
print("CLAIM REWRITE")
print("=" * 70)

print("\nOriginal claim:")
print('"The Decision Tree can accurately identify webpages that need attention."')

print("\nSafer claim:")
print(
    '"In the March 2026 dataset, the Decision Tree showed measured '
    'performance on the selected validation split for the '
    '`needs_attention` proxy. Under grouped-by-client validation, '
    'the result should be treated as directional evidence for '
    'prioritisation rather than proof that the model can identify '
    'every webpage that needs attention."'
)

print("\n✓ Claim changed from a broad capability statement")
print("  to measured, directional, decision-support language.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


CLAIM REWRITE

Original claim:
"The Decision Tree can accurately identify webpages that need attention."

Safer claim:
"In the March 2026 dataset, the Decision Tree showed measured performance on the selected validation split for the `needs_attention` proxy. Under grouped-by-client validation, the result should be treated as directional evidence for prioritisation rather than proof that the model can identify every webpage that needs attention."

✓ Claim changed from a broad capability statement
  to measured, directional, decision-support language.
